# Entrenamiento en Kaggle — U-Net multi-tarea sobre CelebAMask-HQ

Notebook listo para correr en Kaggle con **GPU activada** (Settings → Accelerator → GPU T4 o P100) y el dataset **CelebAMask-HQ** agregado como Input.

Las celdas a continuación:
1. Clonan el repo en `/kaggle/working/`.
2. Localizan automáticamente el dataset en `/kaggle/input/`.
3. Preprocesan las imágenes y combinan las máscaras.
4. Entrenan el modelo.
5. Hacen una demo de inferencia con el checkpoint resultante.

## 1. Clonar el repositorio

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ando0611/ProyectoFinal_IA.git"
WORK = Path("/kaggle/working")
REPO = WORK / "ProyectoFinal_IA"

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("CWD:", os.getcwd())
print("Contenido:")
for p in sorted(REPO.iterdir()):
    print(" ", p.name)

## 2. Localizar el dataset CelebAMask-HQ

Busca automáticamente la carpeta dentro de `/kaggle/input/` que contiene `CelebAMask-HQ-attribute-anno.txt`. Si tienes varios datasets adjuntos, se usa el primero que coincida.

In [ ]:
from pathlib import Path

matches = list(Path("/kaggle/input").rglob("CelebAMask-HQ-attribute-anno.txt"))
assert matches, (
    "No encontré CelebAMask-HQ. Agrega el dataset desde el panel derecho "
    "(Add Input -> CelebAMask-HQ)."
)

RAW = matches[0].parent
print("Raw dataset en:", RAW)

assert (RAW / "CelebA-HQ-img").exists(), "Falta CelebA-HQ-img/"
assert (RAW / "CelebAMask-HQ-mask-anno").exists(), "Falta CelebAMask-HQ-mask-anno/"
print("Estructura OK.")

## 3. Preprocesar

Combina las máscaras binarias por parte en una sola máscara entera 0..18 por imagen y redimensiona a 512×512. Salida en `/kaggle/working/celeba-hq/`.

**`LIMIT`** controla cuántas imágenes procesar. CelebAMask-HQ tiene 30 000 en total; con `5000` el preprocesado tarda ~10–15 min en Kaggle y el entrenamiento ~30–45 min/15 épocas en T4. Sube a `10000`+ si tienes tiempo.

In [ ]:
LIMIT = 5000
OUT = "/kaggle/working/celeba-hq"

!python -m data.preprocess_celebamaskhq --raw "{RAW}" --out {OUT} --limit {LIMIT}

## 4. Entrenar

Las variables de entorno redirigen los outputs (`models/`, `logs/`) a `/kaggle/working/`, que es la única ruta persistente al finalizar la sesión de Kaggle.

Ajusta `EPOCHS`, `BATCH` y `BASE_CH` según GPU disponible:
- **T4 (16 GB)**: `batch=8`, `base_channels=32` funciona bien a `image_size=256`.
- **P100 (16 GB)**: igual.
- Si te quedas sin memoria, baja `batch` a 4 o `base_channels` a 16.

In [ ]:
import os

os.environ["PROJECT_DATA_DIR"] = "/kaggle/working/celeba-hq"
os.environ["PROJECT_MODELS_DIR"] = "/kaggle/working/models"
os.environ["PROJECT_LOGS_DIR"] = "/kaggle/working/logs"

EPOCHS = 15
BATCH = 8
BASE_CH = 32

!python -m src.train --epochs {EPOCHS} --batch-size {BATCH} --base-channels {BASE_CH}

## 5. Inferencia de prueba

Toma algunas imágenes preprocesadas y guarda predicciones (overlay + atributos) en `/kaggle/working/reports/`.

In [ ]:
import os, shutil
from pathlib import Path

os.environ.setdefault("PROJECT_DATA_DIR", "/kaggle/working/celeba-hq")
os.environ.setdefault("PROJECT_MODELS_DIR", "/kaggle/working/models")

# Tomamos 8 imágenes al azar para la demo
src_dir = Path("/kaggle/working/celeba-hq/images")
demo_dir = Path("/kaggle/working/demo_inputs")
demo_dir.mkdir(exist_ok=True)
for p in list(src_dir.glob("*.jpg"))[:8]:
    shutil.copy(p, demo_dir / p.name)

!python -m src.infer \
    --checkpoint /kaggle/working/models/unet_multitask_best.pt \
    --dir /kaggle/working/demo_inputs \
    --save /kaggle/working/reports

In [ ]:
# Visualizar resultados in-line
from pathlib import Path
from IPython.display import Image as IPImage, display

for p in sorted(Path("/kaggle/working/reports").glob("*.jpg")):
    print(p.name)
    display(IPImage(filename=str(p)))

## 6. (Opcional) Descargar el checkpoint

Para no perder el modelo entrenado al cerrar Kaggle:
- En el panel derecho "Output", `/kaggle/working/models/unet_multitask_best.pt` queda disponible para descargar.
- O guarda este notebook como "Save Version" para snapshot completo (modelo + logs + reports).